In [1]:
import numpy as np 
import statsmodels.api as sm
from ISLP.models import (ModelSpec as MS, summarize, poly)
from sklearn.model_selection import train_test_split
import pandas as pd
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from ISLP import load_data

In [2]:
from functools import partial
from sklearn.model_selection import (cross_validate, KFold, ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm 

# The Validation Set Approach

We explore the use of the validation set approach in order to estimate the
test error rates that result from fitting various linear models on the Auto
data set.
We use the function train_test_split() to split the data into training 
and validation sets. As there are 392 observations, we split into two equal
sets of size 196 using the argument test_size=196. It is generally a good
idea to set a random seed when performing operations like this that contain
an element of randomness, so that the results obtained can be reproduced
precisely at a later time. We set the random seed of the splitter with the
argument random_state=0

In [5]:
Auto = load_data('Auto')
Auto_train , Auto_valid = train_test_split(Auto,
                                           test_size=0.5,
                                           random_state=0)


In [6]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

In [7]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
valid_mse = np.mean((y_valid - valid_pred)**2)
print(f'Validation MSE: {valid_mse:.2f}')

Validation MSE: 23.62


We can also estimate the validation error for higher-degree polynomial
regressions. We first provide a function evalMSE() that takes a model string
as well as a training and test set and returns the MSE on the test set.

In [8]:
def evalMSE(terms, response, train, test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]
    
    X_test = mm.transform(test)
    y_test = test[response]
    
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)
    
    return np.mean((y_test - test_pred)**2)

Let’s use this function to estimate the validation MSE using linear,
quadratic and cubic fits. We use the enumerate() function here, which gives enumerate()
both the values and indices of objects as one iterates over a for loop.

In [9]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1,4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
    print('Validation MSE for degrees 1, 2, and 3:', MSE)

Validation MSE for degrees 1, 2, and 3: [23.61661707  0.          0.        ]
Validation MSE for degrees 1, 2, and 3: [23.61661707 18.76303135  0.        ]
Validation MSE for degrees 1, 2, and 3: [23.61661707 18.76303135 18.79694163]


# Cross-Validation
In theory, the cross-validation estimate can be computed for any general
ized linear model. In practice, however, the simplest way to cross-validate
in Python is to use sklearn, which has a different interface or API than
statsmodels, the code we have been using to fit GLMs.
This is a problem which often confronts data scientists: “I have a function
to do task A, and need to feed it into something that performs task B, so
that I can compute B(A(D)), where D is my data.” When A and B don’t
naturally speak to each other, this requires the use of a wrapper. In the ISLP package, we provide a wrapper, sklearn_sm(), that enables us to easily use the cross-validation tools of sklearn with models fit by statsmodels.
The class sklearn_sm() has as its first argument a model from statsmodels.
It can take two additional optional arguments: model_str which can be used
to specify a formula, and model_args which should be a dictionary of addi
tional arguments used when fitting the model. For example, to fit a logistic
regression model we have to specify a family argument. This is passed as
model_args={'family':sm.families.Binomial()}.

In [10]:
hp_model = sklearn_sm(sm.OLS,
                      MS(['horsepower']))
X,Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model,
                            X,
                            Y,
                            cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(24.231513517929212)

The arguments to cross_validate() are as follows: an object with the ap
propriate fit(), predict(), and score() methods, an array of features X and
a response Y. We also included an additional argument cv to cross_validate();
specifying an integer K results in K-fold cross-validation. We have provided
a value corresponding to the total number of observations, which results
in leave-one-out cross-validation (LOOCV). The cross_validate() func- cross_
tion produces a dictionary with several components; we simply want the
cross-validated test score here (MSE), which is estimated to be 24.23.
We can repeat this procedure for increasingly complex polynomial fits.
To automate the process, we again use a for loop which iteratively fits
polynomial regressions of degree 1 to 5, computes the associated cross
validation error, and stores it in the ith element of the vector cv_error.
The variable d in the for loop corresponds to the degree of the polynomial.

In [11]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.42443031, 19.03320428])

In the CV example above, we used K = n, but of course we can also use
K<n.The code is very similar to the above (and is significantly faster).
Here we use KFold() to partition the data into K = 10 random groups. We KFold()
use random_state to set a random seed and initialize a vector cv_error in
which we will store the CV errors corresponding to the polynomial fits of
degrees one to five.


In [12]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0)
for i,d in enumerate(range(1,6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848404, 19.13722016])

In [13]:
validation = ShuffleSplit(n_splits=1,test_size=196,random_state=0)
results = cross_validate(hp_model,Auto.drop(['mpg'], axis=1),Auto['mpg'],cv=validation)
results['test_score']


array([23.61661707])

In [14]:
validation = ShuffleSplit(n_splits=10,
                          test_size=0.5,
                            random_state=0)
results = cross_validate(hp_model,Auto.drop(['mpg'], axis=1),Auto['mpg'],cv=validation)
results['test_score'].mean(), results['test_score'].std()

(np.float64(23.802232661034164), np.float64(1.4218450941091847))